# Story Engine — генератор персонажей для визуальной новеллы

**Anima Aesthetic 1.1 · упрощённая цифровая гуашь · полностью одетые взрослые.**
Модель и стиль согласованы с генератором фонов проекта. По умолчанию — 240 образов:
2 пола × 10 типов одежды × 12 образов на профиль. Включены повседневная,
верхняя, рабочая, дорожная, медицинская и строгая одежда, платье, форма,
одеяние и фантастический комбинезон. 12 групп признаков дают подробные описания,
а эвристика выбирает заметно разные цельные образы из ограниченного пула.
Все сочетания не перебираются; возраст — 25–60. Оценка разнообразия относится
к описаниям: после генерации сравните сами картинки на контактном листе.
Это нейтральные базовые образы: эмоции, гардеробные слои и постоянство лица
между разными генерациями здесь не обещаются.

### Как запустить
1. Загрузите этот `.ipynb` в [Google Colab](https://colab.research.google.com/).
2. **Среда выполнения → Сменить среду выполнения → GPU** (обычно T4).
3. Для всех 240 оставьте `SELECTED_PROFILES="all"`, `COPIES_PER_PROFILE=12`,
   `INCLUDE_EXTRA_OUTFITS=True` и `MAX_IMAGES_THIS_RUN=0`.
   Для первого просмотра задайте `SELECTED_PROFILES = "woman_casual,man_casual"`
   и `COPIES_PER_PROFILE = 1`. Затем **Среда выполнения → Выполнить все**.
4. Разрешите подключение **своего** Google Диска. Результаты появятся в
   `Мой диск/StoryEngine_Characters_Anima/characters_…`.
5. Посмотрите `preview_characters.jpg`, лица, руки и края прозрачности. Для
   полного набора верните `all` и 12 вариантов, оставьте дополнительную одежду включённой,
   повторите ячейку настроек,
   затем разделы **Каталог**, **Генерация**, **Прозрачность и экспорт**.
   При тех же параметрах качества уже готовые образы не перерисуются.

Стартовый размер **640 × 960**, одна картинка за раз, текстовый энкодер и
удаление фона — на CPU, экономия VRAM и плиточное декодирование. Это настройки
под ориентир **11 ГБ VRAM / 12 ГБ RAM**; фактический запуск и качество нужно
проверить на выданной Colab машине. Нейросетевого апскейла здесь нет.

После обрыва Colab подключите GPU и выполните все ячейки снова с теми же
настройками и папкой: целые PNG с подходящими метаданными будут пропущены.
`VARIANT` создаёт другой набор в новой папке. В ZIP входят только картинки,
описания и превью, без весов модели. Автоимпорта в игру нет — сначала отбор.


In [ ]:
#@title Настройки набора персонажей
from pathlib import Path
import hashlib, json, os, sys, time, subprocess, shutil, io, uuid, zipfile
import requests
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

# Полностью одетые взрослые персонажи; один базовый образ, без гардеробных слоёв.
DRIVE_FOLDER = "StoryEngine_Characters_Anima" #@param {type:"string"}
BASE_SEED = 20261020 #@param {type:"integer"}
VARIANT = 1 #@param {type:"integer"}
COPIES_PER_PROFILE = 12 #@param {type:"integer"}
SELECTED_PROFILES = "all" #@param {type:"string"}
INCLUDE_EXTRA_OUTFITS = True #@param {type:"boolean"}
RESOLUTION = "640x960" #@param ["640x960", "768x1152"]
WIDTH, HEIGHT = map(int, RESOLUTION.split("x"))
MAX_IMAGES_THIS_RUN = 0 #@param {type:"integer"}
# 0 — все недостающие образы; например, 4 — короткая партия для просмотра.
STEPS = 40 #@param {type:"integer"}
CFG = 4.5 #@param {type:"number"}
CACHE_WEIGHTS_ON_DRIVE = True #@param {type:"boolean"}
JOB_TIMEOUT_MINUTES = 90 #@param {type:"integer"}
SHOW_PREVIEWS = True #@param {type:"boolean"}
MAKE_ZIP = True #@param {type:"boolean"}
REMOVE_BACKGROUND = True #@param {type:"boolean"}
assert (WIDTH, HEIGHT) in {(640, 960), (768, 1152)}
assert 0 <= MAX_IMAGES_THIS_RUN <= 10000
assert 1 <= COPIES_PER_PROFILE <= 12 and 1 <= VARIANT <= 1000000
assert 1 <= STEPS <= 100 and 1 <= CFG <= 12 and 5 <= JOB_TIMEOUT_MINUTES <= 360
assert DRIVE_FOLDER.strip() and not Path(DRIVE_FOLDER).is_absolute()
assert "/" not in DRIVE_FOLDER and "\\" not in DRIVE_FOLDER, "Задайте имя одной папки MyDrive"
assert ".." not in Path(DRIVE_FOLDER).parts
COMFY_REV = "e0a32bce84f4603f2b61b16b50f8958e68fd6171"
MODEL_REV = "f973fc41ec7545364ac9776c2440285f43ff2a30"
COMFY = Path("/content/ComfyUI_StoryEngine")
VENV = Path("/content/storyengine_venv")
PYTHON = VENV / "bin/python"
API = "http://127.0.0.1:8188"
CLIENT_ID = "storyengine-characters-" + uuid.uuid4().hex


## 1. Установка и подключение личного Google Диска
Первый запуск скачивает около 5,3 ГиБ весов. Если `CACHE_WEIGHTS_ON_DRIVE=True`,
они останутся в `model_cache` для повторного запуска. Эта папка не входит в ZIP.
Установка может занять несколько минут; вывод команд виден под ячейкой.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
ROOT.mkdir(parents=True, exist_ok=True)
print("Личная папка результатов:", ROOT)

def command(args, **kwargs):
    # Печатаем вывод через Python: файловые дескрипторы subprocess в Colab
    # иначе могут уходить в журналы среды вместо выходных данных ячейки.
    from collections import deque
    tail = deque(maxlen=60)
    with subprocess.Popen([str(x) for x in args], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line)
        result = process.wait()
    if result:
        raise RuntimeError(f"Команда завершилась с кодом {result}: {args}\n" + "".join(tail))

# CUDA проверяется отдельным процессом: ядро ноутбука не держит вторую копию PyTorch.
probe_code = '''
import json, torch, importlib.metadata as m
assert torch.cuda.is_available(), 'GPU не подключён. Runtime → Change runtime type → GPU.'
p = torch.cuda.get_device_properties(0)
print(json.dumps({'gpu': p.name, 'vram_gib': p.total_memory / 2**30,
                 'torch': m.version('torch'), 'torchvision': m.version('torchvision')}))
'''
probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError(probe.stderr + "\nВыберите GPU в настройках Colab и повторите эту ячейку.")
GPU_INFO = json.loads(probe.stdout.strip().splitlines()[-1])
mem_kib = int(next(line.split()[1] for line in Path("/proc/meminfo").read_text().splitlines()
                   if line.startswith("MemTotal:")))
print(GPU_INFO, f"ОЗУ: {mem_kib / 2**20:.1f} GiB")
if GPU_INFO["vram_gib"] < 10:
    print("Памяти GPU меньше целевых 11 ГБ; оставьте 640x960.")
if mem_kib / 2**20 < 10:
    print("ОЗУ меньше 10 GiB: CPU-offload может не поместиться.")
if shutil.disk_usage("/content").free < 16 * 2**30:
    raise RuntimeError("Нужно не менее 16 GiB свободного места в /content для среды и весов.")

if not COMFY.exists():
    command(["git", "clone", "--filter=blob:none", "--no-checkout",
             "https://github.com/Comfy-Org/ComfyUI.git", COMFY])
command(["git", "-C", COMFY, "checkout", "--detach", COMFY_REV])
# В Colab может отсутствовать ensurepip (в том числе на Python 3.13).
# Создаём venv без ensurepip и устанавливаем pip через уже имеющийся системный pip.
command([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", VENV])
command([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip"])

# Сохраняем согласованную пару torch/torchvision, предустановленную Colab.
# Изолированная среда используется только сервером, перезапуск ядра не нужен.
constraints = VENV / "storyengine_constraints.txt"
constraints.write_text(f"torch=={GPU_INFO['torch']}\ntorchvision=={GPU_INFO['torchvision']}\n"
                       "transformers==4.57.6\nhuggingface-hub>=0.34.0,<1.0\n", encoding="utf-8")
command([PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
         "-r", COMFY / "requirements.txt", "-c", constraints])
command([PYTHON, "-c", "import torch, torchvision, transformers, spandrel; "
         "assert torch.cuda.is_available(); print('CUDA и зависимости готовы:', torch.__version__)"])
# Фиксируем фактически установленные версии в каталоге каждого запуска позднее.
ENV_FREEZE = subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True)
print("Среда готова.")


## 2. Проверка и загрузка весов Anima
Файлы закреплены по версии и SHA-256, как в блокноте фонов.


In [ ]:
MODEL_FILES = [
    {"path": "split_files/diffusion_models/anima-aesthetic-v1.1.safetensors",
     "kind": "diffusion_models", "size": 4182230656,
     "sha256": "3c1868387a3a1ff504bbb87c33678321965ead381fcf87afbd0264daa600c082"},
    {"path": "split_files/text_encoders/qwen_3_06b_base.safetensors",
     "kind": "text_encoders", "size": 1192135096,
     "sha256": "cd2a512003e2f9f3cd3c32a9c3573f820bb28c940f73c57b1ddaa983d9223eba"},
    {"path": "split_files/vae/qwen_image_vae.safetensors",
     "kind": "vae", "size": 253806246,
     "sha256": "a70580f0213e67967ee9c95f05bb400e8fb08307e017a924bf3441223e023d1f"},
]

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def valid_weight(path, spec):
    return (path.is_file() and path.stat().st_size == spec["size"]
            and file_sha256(path) == spec["sha256"])

def download_weight(spec, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if valid_weight(destination, spec):
        print("Проверен:", destination.name)
        return
    url = spec.get("url") or ("https://huggingface.co/circlestone-labs/Anima/resolve/"
                              + MODEL_REV + "/" + spec["path"])
    partial = destination.with_name(destination.name + ".part")
    for attempt in range(5):
        try:
            offset = partial.stat().st_size if partial.exists() else 0
            if offset >= spec["size"]:
                if valid_weight(partial, spec):
                    os.replace(partial, destination)
                    return
                partial.unlink()
                offset = 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            with requests.get(url, stream=True, headers=headers, timeout=(30, 120)) as r:
                r.raise_for_status()
                resumed = r.status_code == 206 and offset > 0
                if resumed and not r.headers.get("Content-Range", "").startswith(f"bytes {offset}-"):
                    raise RuntimeError("Сервер прислал неверный диапазон скачивания.")
                done = offset if resumed else 0
                last_print = time.monotonic()
                print(f"Загрузка {destination.name}: {done / 2**20:.0f} / {spec['size'] / 2**20:.0f} MiB")
                with open(partial, "ab" if resumed else "wb") as f:
                    for chunk in r.iter_content(4 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            done += len(chunk)
                        if time.monotonic() - last_print > 15:
                            print(f"  {done / spec['size']:.0%}", flush=True)
                            last_print = time.monotonic()
            if not valid_weight(partial, spec):
                partial.unlink(missing_ok=True)
                raise RuntimeError("Размер или SHA256 весов не совпал; файл будет скачан заново.")
            os.replace(partial, destination)
            return
        except (requests.RequestException, RuntimeError) as error:
            if attempt == 4:
                raise
            print("Повтор загрузки:", error)
            time.sleep(min(2 ** (attempt + 1), 30))

WEIGHT_CACHE = ROOT / "model_cache" if CACHE_WEIGHTS_ON_DRIVE else Path("/content/storyengine_model_cache")
for spec in MODEL_FILES:
    target = COMFY / "models" / spec["kind"] / Path(spec["path"]).name
    if valid_weight(target, spec):
        print("Локальный файл проверен:", target.name)
        continue
    cached = WEIGHT_CACHE / MODEL_REV / Path(spec["path"]).name
    download_weight(spec, cached)
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".copying")
    shutil.copyfile(cached, tmp)
    if not valid_weight(tmp, spec):
        raise RuntimeError("Ошибка копирования весов из Drive в /content.")
    os.replace(tmp, target)
print("Все веса готовы и проверены по SHA256.")


## 3. Локальный ComfyUI в режиме экономии памяти
Публичные туннели и платные API не используются.


In [ ]:
# Повтор этой ячейки останавливает только сервер, который запустил этот ноутбук.
if "SERVER" in globals() and SERVER.poll() is None:
    SERVER.terminate()
    try:
        SERVER.wait(timeout=20)
    except subprocess.TimeoutExpired:
        SERVER.kill()
        SERVER.wait(timeout=10)
if "SERVER_LOG_HANDLE" in globals():
    SERVER_LOG_HANDLE.close()
SERVER_LOG = Path("/content/storyengine_comfy.log")
SERVER_LOG_HANDLE = open(SERVER_LOG, "w", encoding="utf-8")
flags = ["--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch",
         "--lowvram", "--reserve-vram", "1.0", "--disable-dynamic-vram",
         "--disable-async-offload", "--disable-pinned-memory", "--cache-none",
         "--fp16-unet", "--fp16-text-enc", "--fp32-vae",
         "--use-pytorch-cross-attention", "--disable-xformers",
         "--disable-all-custom-nodes", "--disable-partner-nodes", "--preview-method", "none"]
SERVER = subprocess.Popen([str(PYTHON), "-u", "main.py", *flags], cwd=COMFY,
                          stdout=SERVER_LOG_HANDLE, stderr=subprocess.STDOUT)

def log_tail():
    if not SERVER_LOG.exists():
        return ""
    with open(SERVER_LOG, "rb") as f:
        f.seek(max(0, SERVER_LOG.stat().st_size - 16000))
        return f.read().decode("utf-8", errors="replace")

deadline = time.monotonic() + 240
while time.monotonic() < deadline:
    if SERVER.poll() is not None:
        raise RuntimeError("ComfyUI не запустился:\n" + log_tail())
    try:
        r = requests.get(API + "/object_info", timeout=5)
        r.raise_for_status()
        NODE_INFO = r.json()
        break
    except requests.RequestException:
        time.sleep(2)
else:
    SERVER.terminate()
    raise RuntimeError("Истекло время запуска ComfyUI:\n" + log_tail())
required_nodes = {"UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "KSampler",
                  "EmptyLatentImage", "VAEDecodeTiled", "SaveImage"}
missing = required_nodes - NODE_INFO.keys()
if missing:
    raise RuntimeError(f"Нет необходимых штатных узлов: {sorted(missing)}\n" + log_tail())
print("Сервер готов. Работает локально, без публичного туннеля.")


## 4. Каталог образов и разнообразие
Общий пул: пол, рост, телосложение, кожа, лицо, глаза, цвет волос, причёска,
растительность на лице, примета, готовый костюм и его основной цвет.
Возраст — отдельное число. Генерируются обычные взрослые люди с простыми
читаемыми лицами; особая примета не обязательна каждому.

На профиль пол/костюм создаются 384 допустимых кандидата с воспроизводимым seed.
Жадный maximin выбирает 12 образов: каждый следующий максимально далёк от
ближайшего уже выбранного в этом профиле по взвешенным признакам.
Возраст, телосложение и причёска важнее глаз и мелких примет. Полная выборка
содержит по четыре образа возрастов 25–34, 35–49, 50–60 на профиль.
Для bald цвет волос отсутствует; для женщин в этой выборке не назначается
растительность на лице, седые/белые волосы предлагаются с 45 лет.
Это эвристика описаний, визуальное отличие картинок проверяется после генерации.

`SELECTED_PROFILES` принимает `all` либо ID через запятую.
`COPIES_PER_PROFILE` — первые 1–12 выбранных образов на профиль.
По умолчанию включены все 10 костюмов: 240 образов.
`INCLUDE_EXTRA_OUTFITS=False` оставляет 6 базовых костюмов: 144 образа.
Все сочетания признаков не перебираются.

Можно начать с 4 вариантов на профиль (80 картинок со всей одеждой, 48 с базовой) и расширить до 12 без
перерисовки уже готовых. Изменение качества, пула или `VARIANT` создаёт новый
набор; старые папки сохраняются. Пул и эвристика синхронизируются из
`backend/character_appearance.py`. Автоматического импорта/подбора спрайта пока нет.


In [ ]:
# BEGIN SHARED CHARACTER APPEARANCE
"""Общий подробный пул внешности и ограниченная разнообразная выборка Colab.

Признаки описывают образ. Эвристика выбирает 12 образов на профиль, не перебирая
декартово произведение и не обещая измеренного визуального отличия рисунков.
"""
import hashlib
import random
from functools import lru_cache

APPEARANCE_POOL_VERSION = 4
APPEARANCE_POOL = {'gender': {'woman': ('женщина', 'woman'), 'man': ('мужчина', 'man')},
 'height': {'short': ('низкий', 'short stature'),
            'average': ('средний', 'average height'),
            'tall': ('высокий', 'tall stature')},
 'build': {'slender': ('стройное', 'slender build'),
           'average': ('обычное', 'average build'),
           'stocky': ('коренастое', 'stocky build'),
           'full': ('полное', 'full-bodied build'),
           'broad': ('широкоплечее', 'broad-shouldered build')},
 'skin': {'very_light': ('очень светлая', 'very light skin'),
          'light': ('светлая', 'light skin'),
          'tan': ('смуглая', 'tan skin'),
          'brown': ('коричневая', 'brown skin'),
          'dark_brown': ('тёмно-коричневая', 'dark brown skin')},
 'face': {'oval': ('овальное', 'oval face'),
          'round': ('круглое', 'round face'),
          'angular': ('угловатое', 'angular face'),
          'long': ('вытянутое', 'long face')},
 'eyes': {'brown': ('карие', 'brown eyes'),
          'hazel': ('ореховые', 'hazel eyes'),
          'gray': ('серые', 'gray eyes'),
          'blue': ('голубые', 'blue eyes'),
          'green': ('зелёные', 'green eyes')},
 'hair_color': {'black': ('чёрные', 'black hair'),
                'dark_brown': ('тёмно-каштановые', 'dark brown hair'),
                'chestnut': ('каштановые', 'chestnut hair'),
                'light_brown': ('русые', 'light brown hair'),
                'blond': ('светлые', 'blond hair'),
                'auburn': ('рыжие', 'auburn hair'),
                'gray': ('седые', 'gray hair'),
                'white': ('белые', 'white hair'),
                'none': ('не применим — без волос', '')},
 'hair': {'cropped': ('очень короткие', 'closely cropped haircut'),
          'short': ('короткие', 'short haircut'),
          'bob': ('каре', 'bob haircut'),
          'shoulder': ('до плеч', 'shoulder-length hairstyle'),
          'long': ('длинные', 'long hairstyle'),
          'ponytail': ('хвост', 'hair tied in a ponytail'),
          'bun': ('пучок', 'hair tied in a bun'),
          'bald': ('без волос', 'bald head')},
 'facial_hair': {'none': ('нет', 'clean-shaven face'),
                 'stubble': ('щетина', 'light stubble'),
                 'mustache': ('усы', 'a simple mustache'),
                 'beard': ('короткая борода', 'a short beard')},
 'detail': {'none': ('нет', ''),
            'freckles': ('веснушки', 'subtle freckles'),
            'mole': ('родинка', 'a small facial mole'),
            'scar': ('небольшой шрам', 'a small facial scar'),
            'glasses': ('простые очки', 'simple eyeglasses')},
 'outfit': {'casual': ('повседневная одежда',
                       'a plain long-sleeved shirt, straight trousers and simple shoes'),
            'coat': ('пальто', 'a modest knee-length coat over a shirt and trousers, closed boots'),
            'work': ('рабочая одежда', 'a woven work shirt, plain trousers, a canvas apron and closed boots'),
            'traveler': ('дорожный костюм',
                         'a layered tunic, a short cloth cape, trousers and leather boots'),
            'medic': ('медицинский халат',
                      'a plain medical coat over everyday clothes and closed shoes, no logos'),
            'formal': ('строгий костюм', 'a restrained jacket, plain shirt, tailored trousers and shoes'),
            'dress': ('простое платье', 'a plain modest long-sleeved dress and closed shoes'),
            'uniform': ('служебная форма', 'a simple fully closed service uniform and boots, no insignia'),
            'robe': ('длинное одеяние',
                     'a plain long cloth robe over an opaque tunic, closed shoes, no symbols'),
            'space': ('фантастический комбинезон',
                      'a simple fully closed science-fiction utility jumpsuit and boots, no helmet')},
 'palette': {'gray': ('серый', 'muted gray'),
             'brown': ('коричневый', 'muted brown'),
             'navy': ('тёмно-синий', 'muted navy blue'),
             'olive': ('оливковый', 'muted olive'),
             'burgundy': ('бордовый', 'muted burgundy'),
             'beige': ('бежевый', 'muted beige')}}
APPEARANCE_LABELS = {'gender': 'Пол',
 'height': 'Рост',
 'build': 'Телосложение',
 'skin': 'Оттенок кожи',
 'face': 'Форма лица',
 'eyes': 'Цвет глаз',
 'hair_color': 'Цвет волос',
 'hair': 'Причёска',
 'facial_hair': 'Растительность на лице',
 'detail': 'Особая примета',
 'outfit': 'Одежда',
 'palette': 'Основной цвет одежды'}
OUTFIT_WORLDS = {'casual': 'modern', 'coat': 'modern', 'work': 'universal', 'traveler': 'historical_fantasy', 'medic': 'modern', 'formal': 'modern', 'dress': 'universal', 'uniform': 'modern', 'robe': 'historical_fantasy', 'space': 'science_fiction'}
BASE_OUTFITS = ('casual', 'coat', 'work', 'traveler', 'medic', 'formal')

LOOKS_PER_PROFILE = 12
DIVERSITY_RECIPE = {
    'version': 2, 'candidates_per_profile': 384,
    'age_bands': ((25, 34), (35, 49), (50, 60)),
    # Максимальный вклад признака в расстояние. Силуэт важнее мелких деталей.
    # gender/outfit фиксированы внутри профиля и покрываются квотами профилей.
    'weights': {'gender': 4, 'age': 3, 'outfit': 5, 'build': 4, 'hair': 4,
                'height': 1.5, 'skin': 2, 'facial_hair': 2,
                'hair_color': 1.5, 'palette': 1, 'face': 0.75, 'detail': 0.5, 'eyes': 0.25},
}


def _number(seed):
    return int.from_bytes(hashlib.sha256(str(seed).encode()).digest()[:8], 'big')


def normalize_appearance(value, *, seed='', gender=None, outfit=None):
    """Дополняет новый ответ локально; старые сохранения сюда не передаются."""
    source = dict(value) if isinstance(value, dict) else {}
    if 'hair' not in source:
        source['hair'] = source.get('hairstyle')
    aliases = {'hair': {'tied': 'ponytail'}, 'skin': {'dark': 'dark_brown'},
               'hair_color': {'dark': 'black', 'light': 'blond'}}
    for group, mapping in aliases.items():
        key = source.get(group)
        if isinstance(key, str):
            source[group] = mapping.get(key, key)
    defaults = {'facial_hair': 'none', 'detail': 'none', 'outfit': 'work'}
    result = {}
    for group, options in APPEARANCE_POOL.items():
        selected = source.get(group)
        if not isinstance(selected, str) or selected not in options:
            keys = [key for key in options if not (group == 'hair_color' and key == 'none')]
            selected = defaults.get(group, keys[_number(f'{seed}:{group}') % len(keys)])
        result[group] = selected
    for group, forced in (('gender', gender), ('outfit', outfit)):
        if isinstance(forced, str) and forced in APPEARANCE_POOL[group]:
            result[group] = forced
    if result['hair'] == 'bald':
        result['hair_color'] = 'none'
    elif result['hair_color'] == 'none':
        colors = [key for key in APPEARANCE_POOL['hair_color'] if key != 'none']
        result['hair_color'] = colors[_number(f'{seed}:hair_color') % len(colors)]
    return result


def describe_appearance(features):
    parts = []
    for group, options in APPEARANCE_POOL.items():
        key = features.get(group)
        if (not isinstance(key, str) or key not in options or key == 'none'
                or group == 'hair_color' and features.get('hair') == 'bald'):
            continue
        text = options[key][0]
        parts.append(text if group == 'gender' else f'{APPEARANCE_LABELS[group].lower()}: {text}')
    return '; '.join(parts)


def appearance_image_prompt(features, age):
    gender = APPEARANCE_POOL['gender'][features['gender']][1]
    parts = [f'{age}-year-old adult {gender}']
    for group in ('height', 'build', 'skin', 'face', 'eyes', 'hair_color', 'hair', 'facial_hair', 'detail'):
        key = features[group]
        if key == 'none' or group == 'hair_color' and features['hair'] == 'bald':
            continue
        parts.append(APPEARANCE_POOL[group][key][1])
    clothes = APPEARANCE_POOL['outfit'][features['outfit']][1]
    palette = APPEARANCE_POOL['palette'][features['palette']][1]
    return ', '.join(parts) + f', wearing {clothes}. Clothing color palette: {palette}.'


def appearance_pool_prompt():
    return '\n'.join(f'{group}: ' + ', '.join(f'{key}={value[0]}' for key, value in options.items())
                     for group, options in APPEARANCE_POOL.items())


def _age_band(age):
    return 0 if age < 35 else 1 if age < 50 else 2


def appearance_distance(first, second):
    """Взвешенная разница описаний, не оценка реального сходства изображений.

    Вход — образ {'age': int, 'appearance_features': {...}}. Соседние оттенки,
    стрижки и возрастные группы различаются слабее разных силуэтов.
    """
    a, b = first['appearance_features'], second['appearance_features']
    weights = DIVERSITY_RECIPE['weights']
    distance = weights['age'] * abs(_age_band(first['age']) - _age_band(second['age'])) / 2
    hair_groups = {'cropped': 'short', 'short': 'short', 'bob': 'medium', 'shoulder': 'medium',
                   'long': 'long', 'ponytail': 'tied', 'bun': 'tied', 'bald': 'bald'}
    color_groups = {'black': 'dark', 'dark_brown': 'dark', 'chestnut': 'dark',
                    'light_brown': 'light', 'blond': 'light', 'auburn': 'red',
                    'gray': 'gray', 'white': 'gray'}
    for group in APPEARANCE_POOL:
        x, y = a.get(group), b.get(group)
        if x == y:
            continue
        difference = 1.0
        if group == 'hair_color':
            if a.get('hair') == 'bald' or b.get('hair') == 'bald':
                continue  # Отсутствие волос уже учтено в причёске.
            if color_groups.get(x) == color_groups.get(y):
                difference = 0.2
        elif group == 'hair' and hair_groups.get(x) == hair_groups.get(y):
            difference = 0.25
        elif group in ('height', 'skin'):
            order = list(APPEARANCE_POOL[group])
            if x in order and y in order:
                difference = abs(order.index(x) - order.index(y)) / (len(order) - 1)
        distance += weights[group] * difference
    return distance


@lru_cache(maxsize=128)
def _profile_looks(gender, outfit, seed):
    """Конечный пул кандидатов + maximin с квотой по взрослым возрастам.

    На каждом шаге берём кандидата, наиболее далёкого от ближайшего выбранного.
    Независимость профилей сохраняет готовые образы при расширении партии.
    """
    if gender not in APPEARANCE_POOL['gender'] or outfit not in APPEARANCE_POOL['outfit']:
        raise ValueError('Неизвестный профиль персонажа')
    rng = random.Random(_number(f'{seed}:{gender}:{outfit}:diversity'))
    candidates, seen = [], set()
    for number in range(DIVERSITY_RECIPE['candidates_per_profile']):
        band = number % len(DIVERSITY_RECIPE['age_bands'])
        age = rng.randint(*DIVERSITY_RECIPE['age_bands'][band])
        source = {group: rng.choice(list(options)) for group, options in APPEARANCE_POOL.items()}
        # Обычные цельные взрослые образы; примета не обязательна каждому.
        source['detail'] = rng.choice(['none'] * 8 + ['freckles', 'mole', 'scar', 'glasses'])
        source['facial_hair'] = (rng.choice(['none'] * 3 + ['stubble', 'mustache', 'beard'])
                                 if gender == 'man' else 'none')
        if age < 45:
            source['hair_color'] = rng.choice(['black', 'dark_brown', 'chestnut', 'light_brown', 'blond', 'auburn'])
        features = normalize_appearance(source, seed=f'{seed}:{number}', gender=gender, outfit=outfit)
        key = (band, tuple(features.items()))
        if key in seen:
            continue
        seen.add(key)
        candidates.append({'age': age, 'appearance_features': features})
    nearest = [float('inf')] * len(candidates)
    selected = []
    available = set(range(len(candidates)))
    for number in range(LOOKS_PER_PROFILE):
        band = number % len(DIVERSITY_RECIPE['age_bands'])
        eligible = [i for i in sorted(available) if _age_band(candidates[i]['age']) == band]
        # При равном балле сохраняем воспроизводимый порядок кандидатов.
        best = max(eligible, key=lambda i: nearest[i])
        chosen = candidates[best]
        selected.append((chosen['age'], tuple(chosen['appearance_features'].items())))
        available.remove(best)
        for i in available:
            nearest[i] = min(nearest[i], appearance_distance(candidates[i], chosen))
    return tuple(selected)


def colab_look(number, *, gender, outfit, seed=''):
    """Новая копия выбранного образа; номер/партия не меняют прошлый префикс."""
    if not isinstance(number, int) or isinstance(number, bool) or not 0 <= number < LOOKS_PER_PROFILE:
        raise ValueError(f'Номер образа должен быть от 0 до {LOOKS_PER_PROFILE - 1}')
    age, features = _profile_looks(gender, outfit, str(seed))[number]
    return {'age': age, 'appearance_features': dict(features)}


def colab_profiles(include_extra=False):
    outfits = APPEARANCE_POOL['outfit'] if include_extra else BASE_OUTFITS
    return [{'id': f'{gender}_{outfit}', 'gender': gender, 'outfit_id': outfit,
             'world': OUTFIT_WORLDS[outfit], 'outfit': APPEARANCE_POOL['outfit'][outfit][1]}
            for gender in APPEARANCE_POOL['gender'] for outfit in outfits]

# END SHARED CHARACTER APPEARANCE


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(",", ":")).encode("utf-8")).hexdigest()

def atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)

def complete_image(path, expected):
    try:
        meta = json.loads(sidecar(path).read_text(encoding="utf-8"))
        if meta.get("signature") != fingerprint(expected):
            return False
        if not path.is_file() or meta.get("image_sha256") != file_sha256(path):
            return False
        with Image.open(path) as im:
            im.load()
            return im.format == "PNG" and im.size == (expected["width"], expected["height"])
    except (OSError, ValueError, KeyError):
        return False

def validate_graph(graph):
    # Проверяем реальную схему установленного сервера, включая доступные sampler/scheduler.
    for key, node in graph.items():
        schema = NODE_INFO[node["class_type"]]
        required = schema.get("input", {}).get("required", {})
        optional = schema.get("input", {}).get("optional", {})
        inputs = node["inputs"]
        missing = set(required) - inputs.keys()
        if missing:
            raise ValueError(f"Узел {key}: не заданы {missing}")
        for name, value in inputs.items():
            if name not in required and name not in optional:
                raise ValueError(f"Узел {key}: неизвестное поле {name}")
            definition = required.get(name, optional.get(name))
            if isinstance(value, list) and len(value) == 2 and isinstance(value[0], str):
                ref, slot = value
                if ref not in graph or not isinstance(slot, int) or slot < 0:
                    raise ValueError(f"Неверная связь {key}.{name}: {value}")
                outputs = NODE_INFO[graph[ref]["class_type"]].get("output", [])
                if slot >= len(outputs):
                    raise ValueError(f"Нет выхода {slot} в узле {ref}")
            elif isinstance(definition[0], list) and value not in definition[0]:
                raise ValueError(f"Узел {key}.{name}: {value!r} не поддерживается сервером")

def api_post(endpoint, body):
    r = requests.post(API + endpoint, json=body, timeout=30)
    if not r.ok:
        raise RuntimeError(f"ComfyUI {endpoint}: {r.status_code}\n{r.text[:8000]}")
    return r.json() if r.content else {}

def diffusion_graph(scene_id, when="day"):
    if when != "day":
        raise ValueError("Этот блок создаёт один базовый образ персонажа.")
    graph = {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "anima-aesthetic-v1.1.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen_3_06b_base.safetensors", "type": "stable_diffusion", "device": "cpu"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt_for(scene_id, when), "clip": ["2", 0]}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"text": negative_for(scene_id, when), "clip": ["2", 0]}},
        "6": {"class_type": "EmptyLatentImage", "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}},
        "7": {"class_type": "KSampler", "inputs": {"seed": scene_seed(scene_id), "steps": STEPS, "cfg": CFG,
              "sampler_name": "er_sde", "scheduler": "simple", "denoise": denoise_for(when),
              "model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0]}},
        "8": {"class_type": "VAEDecodeTiled", "inputs": {"samples": ["7", 0], "vae": ["3", 0],
              "tile_size": 256, "overlap": 64, "temporal_size": 64, "temporal_overlap": 8}},
        "9": {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "storyengine/" + uuid.uuid4().hex}},
    }
    return graph

def stop_job(prompt_id):
    # Этот сервер принадлежит только ноутбуку; отменяем его текущий и ожидающий запрос.
    for endpoint, body in (("/interrupt", {}), ("/queue", {"delete": [prompt_id]})):
        try:
            api_post(endpoint, body)
        except requests.RequestException:
            pass

def run_graph(graph, output_node="9"):
    validate_graph(graph)
    result = api_post("/prompt", {"prompt": graph, "client_id": CLIENT_ID})
    if result.get("node_errors") or "prompt_id" not in result:
        raise RuntimeError("Ошибка графа ComfyUI: " + str(result))
    prompt_id = result["prompt_id"]
    started = time.monotonic()
    last_update = started
    try:
        while time.monotonic() - started < JOB_TIMEOUT_MINUTES * 60:
            if SERVER.poll() is not None:
                raise RuntimeError("Процесс ComfyUI остановился, возможно, закончилась ОЗУ:\n" + log_tail())
            r = requests.get(API + "/history/" + prompt_id, timeout=30)
            r.raise_for_status()
            entry = r.json().get(prompt_id)
            if entry:
                status = entry.get("status", {})
                if status.get("status_str") == "error":
                    raise RuntimeError("Ошибка генерации:\n" + json.dumps(status, ensure_ascii=False)[-8000:]
                                       + "\nЕсли это CUDA out of memory: оставьте RESOLUTION=640x960, "
                                       "повторите настройки и последующие ячейки. Готовые наборы останутся в Drive.\n"
                                       + log_tail())
                images = entry.get("outputs", {}).get(output_node, {}).get("images", [])
                if images and status.get("completed"):
                    asset = images[0]
                    response = requests.get(API + "/view", params=asset, timeout=120)
                    response.raise_for_status()
                    return response.content
                if status.get("completed"):
                    raise RuntimeError("ComfyUI завершил запрос без ожидаемого изображения: " + str(entry)[-8000:])
            if time.monotonic() - last_update > 60:
                print(f"  Генерация идёт: {(time.monotonic() - started) / 60:.1f} мин.", flush=True)
                last_update = time.monotonic()
            time.sleep(2)
        raise TimeoutError(f"Превышен лимит {JOB_TIMEOUT_MINUTES} мин. Увеличьте JOB_TIMEOUT_MINUTES при необходимости.")
    except BaseException:
        try:
            stop_job(prompt_id)
        except Exception:
            pass
        raise

# Каталог и проверенные функции работы с ComfyUI встроены выше.
PROFILES = colab_profiles(INCLUDE_EXTRA_OUTFITS)
SELECTED_PROFILES = ",".join(p.strip() for p in SELECTED_PROFILES.split(",") if p.strip())
selected = PROFILES if SELECTED_PROFILES.strip() == "all" else [p for p in PROFILES if p["id"] in SELECTED_PROFILES.split(",")]
assert selected, "Не выбран ни один профиль"
unknown = set(SELECTED_PROFILES.split(",")) - {p["id"] for p in PROFILES} - {"all"}
assert not unknown, f"Неизвестные профили: {unknown}"
STYLE = ("Digital gouache character illustration with a gentle anime influence, simple readable forms, "
         "broad matte color masses, soft hand-painted texture, restrained detail, harmonious muted colors. "
         "One fully clothed adult person, solo, full body with head and feet visible, centered, "
         "front three-quarter view, relaxed standing pose, arms apart from the torso, empty hands, "
         "neutral attentive expression, soft diffuse daylight, flat plain light gray background, "
         "no scenery, no cast shadow, no ground plane, consistent eye-level camera. ")
NEGATIVE = ("child, teen, young-looking, school uniform, nude, underwear, lingerie, revealing clothes, "
            "photograph, photorealistic, 3d render, pixel art, glossy plastic, highly detailed, "
            "extra limbs, extra fingers, fused hands, cropped head, cropped feet, sitting, "
            "multiple people, duplicate person, crowd, text, watermark, logo, scenery, furniture")
JOBS = []
for profile in selected:
    for number in range(COPIES_PER_PROFILE):
        selected_look = colab_look(number, gender=profile['gender'], outfit=profile['outfit_id'],
                                   seed=f"{BASE_SEED}:{VARIANT}")
        age, features = selected_look['age'], selected_look['appearance_features']
        look = appearance_image_prompt(features, age)
        JOBS.append({**profile, "id":f"{profile['id']}_{number+1:03d}", "age":age,
                     "appearance":describe_appearance(features), "appearance_features":features,
                     "prompt":STYLE + look})
BY_ID = {job["id"]: job for job in JOBS}
ACTIVE_IDS = list(BY_ID)
RECIPE = {"version":6,"purpose":"fully-clothed-character-library", "model_revision":MODEL_REV,
          "comfy_revision":COMFY_REV,"models":MODEL_FILES,"base_seed":BASE_SEED,"variant":VARIANT,
          "width":WIDTH,"height":HEIGHT,"steps":STEPS,"cfg":CFG,"style":STYLE,"negative":NEGATIVE,
          "appearance_pool_version":APPEARANCE_POOL_VERSION,"appearance_pool":APPEARANCE_POOL,"diversity":DIVERSITY_RECIPE}
RUN_HASH = fingerprint(RECIPE)
RUN = ROOT / ("characters_" + RUN_HASH[:16])
ORIGINALS, CUTOUTS, PREVIEWS = RUN/"originals", RUN/"cutouts", RUN/"previews"
for folder in [ORIGINALS,CUTOUTS,PREVIEWS]: folder.mkdir(parents=True, exist_ok=True)
atomic_json(RUN/"recipe.json",RECIPE)
atomic_json(RUN/"environment.json", {"gpu":globals().get("GPU_INFO"),
                                    "comfy_flags":globals().get("flags", []),
                                    "packages":globals().get("ENV_FREEZE", "not recorded")})
atomic_json(RUN/"last_selection.json", {"ids":ACTIVE_IDS,"copies_per_profile":COPIES_PER_PROFILE,
                                      "selected_profiles":SELECTED_PROFILES})

def scene_seed(scene_id):
    return int.from_bytes(hashlib.sha256(f"{BASE_SEED}:{VARIANT}:{scene_id}".encode()).digest()[:8],"big") % (2**63-1)
def prompt_for(scene_id, when="day"):
    return BY_ID[scene_id]["prompt"]
def negative_for(scene_id, when="day"):
    return NEGATIVE
def denoise_for(when):
    return 1.0
def original_path(scene_id):
    return ORIGINALS / f"{scene_id}.png"
def sidecar(path):
    return path.with_suffix(".json")
def original_expected(scene_id):
    return {"run_hash":RUN_HASH,"id":scene_id,"seed":scene_seed(scene_id),"prompt":prompt_for(scene_id),
            "width":WIDTH,"height":HEIGHT,"negative":NEGATIVE,"character":BY_ID[scene_id]}
def commit_image(path, png_bytes, expected):
    with Image.open(io.BytesIO(png_bytes)) as image:
        image.load()
        assert image.size == (WIDTH,HEIGHT), image.size
        image = image.convert("RGB")
    temporary = path.with_suffix(".tmp")
    image.save(temporary,format="PNG")
    os.replace(temporary,path)
    atomic_json(sidecar(path),{"signature":fingerprint(expected),"parameters":expected,"image_sha256":file_sha256(path)})

print(f"Выбрано {len(ACTIVE_IDS)} полностью одетых персонажей. Папка: {RUN}")

def catalog_originals():
    # Включаем все проверенные изображения этого набора, в том числе
    # сгенерированные ранее с другим выбором профилей/числом вариантов.
    entries = []
    for path in sorted(ORIGINALS.glob("*.png")):
        try:
            parameters = json.loads(sidecar(path).read_text(encoding="utf-8"))["parameters"]
            if parameters.get("run_hash") != RUN_HASH or not complete_image(path, parameters):
                continue
            character = parameters["character"]
            entries.append({**character, "original":path.relative_to(RUN).as_posix(),
                            "seed":parameters["seed"], "sha256":file_sha256(path), "approved":False})
        except (OSError, ValueError, KeyError, TypeError):
            print("Пропущен повреждённый исходник или JSON:", path.name)
    return entries

def current_cutout(source):
    path = CUTOUTS / source.name
    try:
        meta = json.loads(sidecar(path).read_text(encoding="utf-8"))
        if (meta.get("source_sha256") != file_sha256(source)
                or meta.get("image_sha256") != file_sha256(path)
                or meta.get("method") != "rembg-u2netp-v1"):
            return None
        with Image.open(path) as image:
            image.load()
            if image.mode != "RGBA" or image.size != (WIDTH, HEIGHT):
                return None
            low, high = image.getchannel("A").getextrema()
            if low == 255 or high == 0:
                return None
        return path
    except (OSError, ValueError, KeyError):
        return None


In [ ]:
# 5. Генерация. Каждый готовый исходник сразу сохраняется на личном Drive.
generated = 0
for index, scene_id in enumerate(ACTIVE_IDS, 1):
    path, expected = original_path(scene_id), original_expected(scene_id)
    if complete_image(path, expected):
        print(f"[{index}/{len(ACTIVE_IDS)}] Уже готов: {scene_id}")
        continue
    if MAX_IMAGES_THIS_RUN and generated >= MAX_IMAGES_THIS_RUN:
        print("Лимит партии достигнут; оставшиеся образы — при следующем запуске этой ячейки.")
        break
    print(f"[{index}/{len(ACTIVE_IDS)}] Рисуем: {scene_id}", flush=True)
    commit_image(path, run_graph(diffusion_graph(scene_id)), expected)
    generated += 1
ready = sum(complete_image(original_path(i), original_expected(i)) for i in ACTIVE_IDS)
print(f"За эту партию создано: {generated}; готово в текущем выборе: {ready}/{len(ACTIVE_IDS)}.")
print("Исходники сохранены:", ORIGINALS)


## 6. Прозрачный фон, превью и архив
Сначала Anima освобождает память. Маленькая `u2netp` удаляет серый фон на CPU.
Это автоматическая маска: волосы, кисти и края одежды требуют визуального отбора.
Исходники остаются в `originals`, вырезки — в `cutouts`. Уже готовые корректные
вырезки повторно не обрабатываются. Если отключить `REMOVE_BACKGROUND`, каталог
и архив будут содержать исходники без прозрачности.

ZIP собирается даже для неполного набора и содержит только проверенные готовые
файлы. В каталоге `approved=false` — изображения ещё нужно просмотреть.
Эта ячейка сохраняет все готовые образы текущего набора, включая предыдущие партии.
После генерации скачайте **ZIP**, а не папку `model_cache` с весами.


In [ ]:
def export_characters():
    entries = catalog_originals()
    if not entries:
        print("Готовых проверенных исходников пока нет. Запустите ячейку генерации.")
        return None
    # Выгружаем диффузию перед CPU-постобработкой даже при отключённой маске.
    api_post("/free", {"unload_models": True, "free_memory": True})
    if REMOVE_BACKGROUND:
        command([PYTHON, "-m", "pip", "install", "rembg==2.0.67", "onnxruntime==1.22.1"])
        worker = Path("/content/storyengine_remove_characters.py")
        worker.write_text('"""Worker Colab: маленькая модель маски на CPU, оригиналы не изменяются."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport sys\nimport uuid\nfrom PIL import Image\nfrom rembg import new_session, remove\n\nmanifest = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\nroot = Path(manifest[\'root\']).resolve()\nsession = new_session(\'u2netp\', providers=[\'CPUExecutionProvider\'])\n\ndef sha(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\nfor relative in manifest[\'originals\']:\n    source = (root / relative).resolve()\n    if not source.is_relative_to(root / \'originals\') or source.suffix != \'.png\':\n        raise ValueError(\'Недопустимый путь исходника\')\n    target = root / \'cutouts\' / source.name\n    metadata = target.with_suffix(\'.json\')\n    try:\n        source_sha = sha(source)\n        try:\n            existing = json.loads(metadata.read_text(encoding=\'utf-8\'))\n            if (existing.get(\'source_sha256\') == source_sha\n                    and existing.get(\'image_sha256\') == sha(target)\n                    and existing.get(\'method\') == \'rembg-u2netp-v1\'):\n                with Image.open(target) as image:\n                    image.load()\n                    alpha = image.getchannel(\'A\').getextrema() if image.mode == \'RGBA\' else (255, 255)\n                    if image.size == (manifest[\'width\'], manifest[\'height\']) and alpha[0] < 255 and alpha[1] > 0:\n                        print(\'Вырезка уже готова:\', target.name, flush=True)\n                        continue\n        except (OSError, ValueError, KeyError):\n            pass\n        result = remove(source.read_bytes(), session=session)\n        temporary = target.with_name(target.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n        temporary.write_bytes(result)\n        with Image.open(temporary) as image:\n            image.load()\n            if image.mode != \'RGBA\' or image.size != (manifest[\'width\'], manifest[\'height\']):\n                raise ValueError(\'Неверный формат вырезки\')\n            low, high = image.getchannel(\'A\').getextrema()\n            if low == 255 or high == 0:\n                raise ValueError(\'Маска полностью непрозрачная или пустая\')\n        os.replace(temporary, target)\n        data = {\'source_sha256\':source_sha, \'image_sha256\':sha(target), \'method\':\'rembg-u2netp-v1\'}\n        meta_temp = metadata.with_name(metadata.name + \'.tmp\')\n        meta_temp.write_text(json.dumps(data), encoding=\'utf-8\')\n        os.replace(meta_temp, metadata)\n        print(\'Вырезка сохранена:\', target.name, flush=True)\n    except Exception as error:\n        print(\'Маска не получилась, исходник сохранён:\', source.name, str(error), flush=True)\n', encoding="utf-8")
        worklist = RUN / "background_removal_jobs.json"
        atomic_json(worklist, {"root":str(RUN), "originals":[e["original"] for e in entries],
                              "width":WIDTH, "height":HEIGHT})
        cache = ROOT / "model_cache" / "u2net"
        cache.mkdir(parents=True, exist_ok=True)
        command([PYTHON, worker, worklist], env={**os.environ, "U2NET_HOME":str(cache),
                                              "OMP_NUM_THREADS":"2", "OPENBLAS_NUM_THREADS":"2"})

    for item in entries:
        cutout = current_cutout(RUN / item["original"]) if REMOVE_BACKGROUND else None
        item.update({"cutout":cutout.relative_to(RUN).as_posix() if cutout else None,
                     "cutout_sha256":file_sha256(cutout) if cutout else None,
                     "width":WIDTH, "height":HEIGHT, "expression":"neutral"})
    missing = [i for i in ACTIVE_IDS if not complete_image(original_path(i), original_expected(i))]
    manifest = {"version":2, "style":"digital_gouache", "run_hash":RUN_HASH,
                "selected_ids":ACTIVE_IDS, "missing_selected":missing,
                "characters":entries}
    atomic_json(RUN / "character_catalog.json", manifest)
    columns, thumb_w, thumb_h = 6, 192, 312
    sheet = Image.new("RGB", (columns * thumb_w, ((len(entries) + columns - 1) // columns) * thumb_h), "#b9b2a6")
    draw = ImageDraw.Draw(sheet)
    for index, item in enumerate(entries):
        x, y = index % columns * thumb_w, index // columns * thumb_h
        for ty in range(0, thumb_h - 24, 16):
            for tx in range(0, thumb_w, 16):
                color = "#dddddd" if (tx // 16 + ty // 16) % 2 else "#aaaaaa"
                draw.rectangle((x + tx, y + ty, x + min(tx + 15, thumb_w - 1),
                                y + min(ty + 15, thumb_h - 25)), fill=color)
        with Image.open(RUN / (item["cutout"] or item["original"])) as image:
            thumb = ImageOps.contain(image.convert("RGBA"), (thumb_w, thumb_h - 24))
        sheet.paste(thumb, (x + (thumb_w - thumb.width) // 2, y), thumb)
        draw.text((x + 3, y + thumb_h - 20), item["id"], fill="#202020")
    sheet.save(RUN / "preview_characters.jpg", quality=93)
    if SHOW_PREVIEWS:
        display(sheet)
    readme = RUN / "CHARACTERS_README.txt"
    readme.write_text(
        "Story Engine: полностью одетые взрослые персонажи, цифровая гуашь.\n"
        "originals/: исходные PNG и параметры; cutouts/: PNG с прозрачностью и параметры маски.\n"
        "character_catalog.json: внешность, костюм, мир, seed, пути и контрольные суммы.\n"
        "preview_characters.jpg: обзор; approved=false до визуального отбора.\n"
        "Проверьте лица, руки, ступни, костюм и вырезание волос; это не автоматическое одобрение качества.\n"
        "Прямой импорт/показ персонажей в игре этим блокнотом не добавляется.\n"
        "Веса модели и кэш в архив не включены.\n"
        "Anima: https://huggingface.co/circlestone-labs/Anima\n"
        "ComfyUI: https://github.com/Comfy-Org/ComfyUI\n"
        "rembg: https://github.com/danielgatis/rembg\n"
        "u2net: https://github.com/xuebinqin/U-2-Net\n",
        encoding="utf-8")
    archive = None
    if MAKE_ZIP:
        archive = ROOT / (RUN.name + ".zip")
        temporary = archive.with_name(archive.name + ".tmp")
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_STORED) as output:
            files = {RUN / item["original"] for item in entries}
            files |= {sidecar(path) for path in list(files)}
            cutouts = {RUN / item["cutout"] for item in entries if item["cutout"]}
            files |= cutouts | {sidecar(path) for path in cutouts}
            files |= {RUN / name for name in ["recipe.json", "character_catalog.json",
                                              "environment.json", "preview_characters.jpg", "CHARACTERS_README.txt"]}
            for path in sorted(files):
                output.write(path, path.relative_to(RUN).as_posix())
        os.replace(temporary, archive)
    print(f"Сохранено образов этого набора: {len(entries)}; с прозрачностью: {sum(bool(e['cutout']) for e in entries)}.")
    print("Папка:", RUN)
    if archive:
        print(f"Скачайте ZIP ({archive.stat().st_size / 2**20:.1f} MiB):", archive)
    if missing:
        print(f"Выбранная партия пока не полная: осталось {len(missing)}. Повторите ячейку генерации.")
    return manifest


manifest = export_characters()
